In [1]:
import numpy as np
import pandas as pd
import csv
import random
from sklearn.linear_model import LogisticRegression
import sys
import os

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
random.seed(0)
np.random.seed(0)

from utils import *

enc = 'utf-8'

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'CMacedonia'

In [4]:
YEAR = 2023

In [5]:
base_path = f'../../data/{NUTS2}/results/'

In [6]:
file_list = os.listdir(base_path)
file_list.sort()
file_list

['GR_CMacedonia_Results_2010-2022.csv',
 'GR_CMacedonia_Results_2023-July-1st.csv',
 'GR_CMacedonia_Results_2023-July-2nd.csv',
 'GR_CMacedonia_Results_2023-June-1st.csv',
 'GR_CMacedonia_Results_2023-June-2nd.csv',
 'GR_CMacedonia_Results_2023-May-1st.csv',
 'GR_CMacedonia_Results_2023-May-2nd.csv']

In [7]:
result_files = []

for file in file_list:
    data = pd.read_csv(f'{base_path}/{file}', encoding='utf-8')
    data.drop(columns=['x', 'y'], inplace=True)
    data.rename(columns={"probability": "score", "municipality": "lau1"}, inplace=True)
    result_files.append(data)

In [8]:
result_files[0] = result_files[0].loc[(result_files[0]['year'] == 2022)]

In [9]:
result_files[0].drop(columns = ['case'], inplace = True)

In [10]:
result_files[1].drop(columns = ['risk_class'], inplace = True)

In [11]:
result_files[2].drop(columns = ['risk_class'], inplace = True)

In [12]:
result_files[3].drop(columns = ['Class Number - (Relative Risk Level)', 'Class Number - (Risk Level)'], inplace = True)

In [13]:
result_files[4].drop(columns = ['Class Number - (Relative Risk Level)', 'Class Number - (Risk Level)'], inplace = True)

In [14]:
result_files[5].drop(columns = ['Class Number - (Relative Risk Level)', 'Class Number - (Risk Level)'], inplace = True)

In [15]:
result_files[6].drop(columns = ['Class Number - (Relative Risk Level)', 'Class Number - (Risk Level)'], inplace = True)

In [16]:
concated = pd.concat(result_files, ignore_index=True, axis = 0)

In [17]:
concated.month.unique()

array([ 8,  9,  7, 10,  6, 11,  5], dtype=int64)

In [18]:
zero_score = {'lau1': 'αγνωστη', 'day': 0, 'month': 0, 'year': 0, 'score': 0}
one_score = {'lau1': 'αγνωστη', 'day': 0, 'month': 0, 'year': 0, 'score': 1}

In [19]:
concated

,lau1,day,month,year,score
0,βεροιας,1,8,2022,0.976101
1,κατερινης,1,8,2022,0.973144
2,σκυδρας,1,8,2022,0.973015
3,θεσσαλονικης,16,8,2022,0.972536
4,βεροιας,16,8,2022,0.968615
...,...,...,...,...,...
679,πυδνας κολινδρου,16,5,2023,0.001364
680,νεαπολης συκεων,16,5,2023,0.001020
681,πολυγυρου,16,5,2023,0.000984
682,αριστοτελη,16,5,2023,0.000667


In [20]:
concated_og = concated.copy()

In [21]:
high_values_rows = concated_og.loc[concated_og['month'].isin([7, 8])]
concated = pd.concat([concated, high_values_rows], ignore_index=True)

In [22]:
high_score_rows = concated_og.loc[concated_og['score'] >= 0.8]
very_high_score_rows = concated_og.loc[concated_og['score'] >= 0.9]

concated = pd.concat([concated, high_score_rows], ignore_index=True)
concated = pd.concat([concated, high_score_rows], ignore_index=True)
concated = pd.concat([concated, very_high_score_rows], ignore_index=True)

In [23]:
concated = concated.append(zero_score, ignore_index = True)
concated = concated.append(one_score, ignore_index = True)

In [24]:
concated

,lau1,day,month,year,score
0,βεροιας,1,8,2022,0.976101
1,κατερινης,1,8,2022,0.973144
2,σκυδρας,1,8,2022,0.973015
3,θεσσαλονικης,16,8,2022,0.972536
4,βεροιας,16,8,2022,0.968615
...,...,...,...,...,...
1279,πυλαιας χορτιατη,16,7,2023,0.925868
1280,κασσανδρας,16,7,2023,0.904282
1281,θεσσαλονικης,16,7,2023,0.903095
1282,αγνωστη,0,0,0,0.000000


In [25]:
out, bins = pd.qcut(concated['score'], 10, retbins= True, labels=range(10))

In [26]:
concated_og['risk_class'] = pd.cut(concated_og['score'], bins=bins, labels=False, include_lowest=True)

In [27]:
concated_og.risk_class.value_counts()

0    128
1    127
2    112
3     93
4     75
5     37
6     34
7     27
8     26
9     25
Name: risk_class, dtype: int64

In [28]:
bins_formatted = [ '%.4f' % elem for elem in bins]
print(bins_formatted)

['0.0000', '0.0046', '0.0685', '0.2636', '0.5802', '0.7882', '0.8579', '0.9012', '0.9371', '0.9622', '1.0000']


In [29]:
scores = concated['score'].tolist()

In [30]:
bins = bins.tolist()

In [31]:
scores_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Scores_{YEAR}_(8).csv'

with open(scores_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(scores)

In [32]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}_(8).csv'

with open(bins_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(bins)